In [5]:
# Video processor class
class VideoProcessor:
    def __init__(self):
        self.counter = 0
        self.stage = None
        self.pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)
        self.last_update_time = time.time()
        
    def recv(self, frame):
        img = frame.to_ndarray(format="bgr24")
        
        # Process with MediaPipe
        image = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = self.pose.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        try:
            landmarks = results.pose_landmarks.landmark
            
            # Get coordinates
            shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,
                       landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
            elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x,
                    landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
            wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x,
                    landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]
            
            # Calculate angle
            angle = calculate_angle(shoulder, elbow, wrist)
            
            # Visualize angle
            coords = tuple(np.multiply(elbow, [img.shape[1], img.shape[0]]).astype(int))
            cv2.putText(image, f"{int(angle)}°", coords,
                       cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2, cv2.LINE_AA)
            
            # Curl counter logic
            if angle > 160:
                self.stage = "down"
            if angle < 35 and self.stage == "down":
                self.stage = "up"
                self.counter += 1
                self.last_update_time = time.time()
                
        except Exception as e:
            print("Error calculating angle:", e)
        
        # Draw pose landmarks - FIXED THIS SECTION
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                image, 
                results.pose_landmarks, 
                mp_pose.POSE_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=2),
                mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2)
            )
        
        # Rest of the code remains the same...
        # Create status box
        cv2.rectangle(image, (0, 0), (300, 100), (32, 32, 32), -1)
        
        # Reps data
        cv2.putText(image, 'REPS', (15, 25), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 1, cv2.LINE_AA)
        cv2.putText(image, str(self.counter), 
                   (15, 70), 
                   cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 2, cv2.LINE_AA)
        
        # Stage data
        cv2.putText(image, 'STAGE', (120, 25), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 1, cv2.LINE_AA)
        stage_text = self.stage if self.stage else "N/A"
        cv2.putText(image, stage_text, 
                   (120, 70), 
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
        
        # Add watermark
        cv2.putText(image, "AI Fitness Coach", (image.shape[1]-200, image.shape[0]-20), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        
        return av.VideoFrame.from_ndarray(image, format="bgr24")

In [ ]:
!streamlit run app.ipynb